# Varying regularization in Multi-layer Perceptron: sklearn vs Moxie

**Reference example:** `../scikit-learn/examples/neural_networks/plot_mlp_alpha.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([rendered gallery page](https://scikit-learn.org/dev/auto_examples/neural_networks/plot_mlp_alpha.html)).

**Scenario:** `make_pipeline(StandardScaler(), MLPClassifier(solver="lbfgs", alpha=alpha, random_state=1,
max_iter=2000, early_stopping=True, hidden_layer_sizes=[10, 10]))` for five `alpha` values on the moons, circles and
a linearly separable dataset, with `predict_proba` decision surfaces and test scores.

**Moxie status:** Dense L-BFGS training, classification, regression and
probability prediction are implemented. The Mojo pipeline uses the same datasets,
splits and model configuration. The side-by-side diagnostics cover all
full-budget panels. A six-iteration replay checks pipeline parity before
the solver paths diverge on some of the full-budget fits. Both implementations ignore
`early_stopping=True` for L-BFGS; stochastic validation-set stopping is separate.

**Adaptations:** smaller figure (dpi 60). Datasets are generated in Python and shared.

**Determinism:** deterministic (fixed dataset seeds, `random_state=1`, full-batch L-BFGS).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("neural_network/plot_mlp_alpha")

## 1. scikit-learn (Python reference)

The gallery code uses the pinned estimator classes and records the fitted probabilities, labels and scores for comparison.

A shorter L-BFGS fit with the same initialization and data gives a solver parity check for each panel.

In [2]:
import runpy

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_circles, make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

reference = runpy.run_path(str(ROOT / "families/neural_network/tests/mlp_parity.py"))["load_reference"]()
MLPClassifier, MLPRegressor = reference.MLPClassifier, reference.MLPRegressor

h = 0.02  # step size in the mesh

alphas = np.logspace(-1, 1, 5)

classifiers = []
names = []
for alpha in alphas:
    classifiers.append(
        make_pipeline(
            StandardScaler(),
            MLPClassifier(
                solver="lbfgs",
                alpha=alpha,
                random_state=1,
                max_iter=2000,
                early_stopping=True,
                hidden_layer_sizes=[10, 10],
            ),
        )
    )
    names.append(f"alpha {alpha:.2f}")

X, y = make_classification(
    n_features=2, n_redundant=0, n_informative=2, random_state=0, n_clusters_per_class=1
)
rng = np.random.RandomState(2)
X += 2 * rng.uniform(size=X.shape)
linearly_separable = (X, y)

datasets = [
    make_moons(noise=0.3, random_state=0),
    make_circles(noise=0.2, factor=0.5, random_state=1),
    linearly_separable,
]

figure = plt.figure(figsize=(12, 6), dpi=60)
i = 1
splits, scores, scaled = [], [], []
predictions, probabilities, mesh_probabilities, grids = [], [], [], []
# iterate over datasets
for X, y in datasets:
    # split into training and test part
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.4, random_state=42
    )
    splits.append((X_train, X_test, y_train, y_test))

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    grid = np.column_stack([xx.ravel(), yy.ravel()])
    grids.append(grid)

    # just plot the dataset first
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(["#FF0000", "#0000FF"])
    ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
    # Plot the training points
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright)
    # and testing points
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, alpha=0.6)
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.set_xticks(())
    ax.set_yticks(())
    i += 1

    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)
        scores.append(score)
        predictions.append(clf.predict(X_test))
        probabilities.append(clf.predict_proba(X_test))

        # Plot the decision boundary. For that, we will assign a color to each
        # point in the mesh [x_min, x_max] x [y_min, y_max].
        if hasattr(clf, "decision_function"):
            Z = clf.decision_function(grid)
        else:
            Z = clf.predict_proba(grid)[:, 1]

        mesh_probabilities.append(Z.copy())

        # Put the result into a color plot
        Z = Z.reshape(xx.shape)
        ax.contourf(xx, yy, Z, cmap=cm, alpha=0.8)

        # Plot also the training points
        ax.scatter(
            X_train[:, 0],
            X_train[:, 1],
            c=y_train,
            cmap=cm_bright,
            edgecolors="black",
            s=25,
        )
        # and testing points
        ax.scatter(
            X_test[:, 0],
            X_test[:, 1],
            c=y_test,
            cmap=cm_bright,
            alpha=0.6,
            edgecolors="black",
            s=25,
        )

        ax.set_xlim(xx.min(), xx.max())
        ax.set_ylim(yy.min(), yy.max())
        ax.set_xticks(())
        ax.set_yticks(())
        ax.set_title(name)
        ax.text(
            xx.max() - 0.3,
            yy.min() + 0.3,
            f"{score:.3f}".lstrip("0"),
            size=15,
            horizontalalignment="right",
        )
        i += 1

figure.subplots_adjust(left=0.02, right=0.98)
plt.show()
print(np.array(scores).reshape(3, 5))
for k, (X_train, X_test, y_train, y_test) in enumerate(splits):
    scaled.append(StandardScaler().fit(X_train).transform(X_test))
    nb.share(**{f"X_train{k}": X_train, f"X_test{k}": X_test,
                f"y_train{k}": y_train.astype(np.int64), f"y_test{k}": y_test.astype(np.int64),
                f"grid{k}": grids[k]})
nb.share(alphas=alphas)

bounded_probabilities = []
for X_train, X_test, y_train, _ in splits:
    for alpha in alphas:
        bounded = make_pipeline(StandardScaler(), MLPClassifier(
            hidden_layer_sizes=[10, 10], solver="lbfgs", alpha=alpha, random_state=1,
            max_iter=6, early_stopping=True,
        ))
        bounded.fit(X_train, y_train)
        bounded_probabilities.append(bounded.predict_proba(X_test))

X_train, X_test, y_train, _ = splits[0]
regression = MLPRegressor(hidden_layer_sizes=[10, 10], random_state=1, max_iter=20)
regression.fit(X_train, y_train.astype(np.float64))
regression_predictions = regression.predict(X_test)

[[0.875 0.95  0.95  0.875 0.475]
 [0.8   0.875 0.9   0.4   0.4  ]
 [0.875 0.95  0.95  0.925 0.825]]


/tmp/ipykernel_1946013/3730703283.py:140: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/strgenix/postdoc/projects/mojods/porting-sklearn/scikit-learn/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 6 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=6).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/strgenix/postdoc/projects/mojods/porting-sklearn/scikit-learn/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 6 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=6).
You might also want to scale the data as sho

## 2. Moxie (native Mojo)

The first check compares StandardScaler on the same three splits.

In [3]:
%%mojo scaler
from moxie.preprocessing import StandardScaler
from moxie_nb_io import load, save


def main() raises:
    for k in range(3):
        var scaler = StandardScaler()
        _ = scaler.fit(load("X_train" + String(k)))
        save("scaled" + String(k), scaler.transform(load("X_test" + String(k))))

### L-BFGS classifier pipeline

This cell fits the native pipeline for all three datasets and five regularization strengths.

It also repeats every fit with a six-iteration limit for numerical comparison.

In [4]:
%%mojo mlp_pipeline
from numojo import Shape, zeros
from moxie.neural_network import MLPClassifier
from moxie.pipeline import make_pipeline
from moxie.preprocessing import StandardScaler
from moxie_nb_io import load, save


def main() raises:
    var alphas = load("alphas")
    var scores = zeros[DType.float64](Shape(3, alphas.size))
    for k in range(3):
        var train = load("X_train" + String(k))
        var test = load("X_test" + String(k))
        var y_train = load[DType.int64]("y_train" + String(k))
        var y_test = load[DType.int64]("y_test" + String(k))
        var grid = load("grid" + String(k))
        for a in range(alphas.size):
            var pipe = make_pipeline(StandardScaler(), MLPClassifier(
                [10, 10], solver="lbfgs", alpha=alphas.unsafe_get(a), random_state=1,
                max_iter=2000, early_stopping=True,
            ))
            _ = pipe.fit(train, y_train)
            var panel = k * alphas.size + a
            scores.unsafe_set(panel, pipe.score(test, y_test))
            save("labels" + String(panel), pipe.predict(test))
            save("proba" + String(panel), pipe.predict_proba(test))
            var surface = pipe.predict_proba(grid)
            var positive = zeros[DType.float64](Shape(grid.shape[0]))
            for row in range(positive.size):
                positive.unsafe_set(row, surface.unsafe_get(2 * row + 1))
            save("mesh" + String(panel), positive)
            var params = pipe.get_params()
            params.estimator.max_iter = 6
            _ = pipe.set_params(params).fit(train, y_train)
            save("bounded" + String(panel), pipe.predict_proba(test))
    save("scores", scores)

### Regressor

A seeded Adam fit checks the regressor's scalar-target prediction path on the first split.

In [5]:
%%mojo mlp_regressor
from moxie.neural_network import MLPRegressor
from moxie_nb_io import load, save


def main() raises:
    var X = load("X_train0")
    var y = load[DType.int64]("y_train0").astype[DType.float64]()
    var model = MLPRegressor(hidden_layer_sizes=[10, 10], random_state=1, max_iter=20)
    _ = model.fit(X, y)
    save("predictions", model.predict(load("X_test0")))

## 3. Side by side and parity

In [6]:
mx = nb.results["scaler"]
moxie_nb.side_by_side(sklearn=scaled[0][:5], moxie=mx["scaled0"][:5])
for k, name in enumerate(["moons", "circles", "linearly separable"]):
    nb.compare(f"StandardScaler step ({name})", scaled[k], mx[f"scaled{k}"], rtol=1e-12, atol=1e-14,
               reason="Float64 mean and variance reductions")

native = nb.results["mlp_pipeline"]
from IPython.display import Markdown, display

report = [
    "| Dataset | α | sklearn accuracy | Moxie accuracy | Test label differences | Max test probability difference | Max mesh probability difference |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for panel in range(len(scores)):
    k, a = divmod(panel, len(alphas))
    name = ("Moons", "Circles", "Linear")[k]
    label_difference = np.count_nonzero(predictions[panel] != native[f"labels{panel}"])
    test_difference = np.max(np.abs(probabilities[panel] - native[f"proba{panel}"]))
    mesh_difference = np.max(np.abs(mesh_probabilities[panel] - native[f"mesh{panel}"]))
    report.append(
        f"| {name} | {alphas[a]:.3g} | {scores[panel]:.3f} | {native['scores'][k, a]:.3f} | "
        f"{label_difference} | {test_difference:.3g} | {mesh_difference:.3g} |"
    )
    nb.compare(f"Panel {panel + 1} six-iteration test probabilities",
               bounded_probabilities[panel], native[f"bounded{panel}"], rtol=2e-10, atol=2e-11,
               reason="Float64 reductions and six L-BFGS iteration updates")

display(Markdown("\n".join(report)))
nb.note("Full-budget classifier panels", "GAP",
        "The table reports exact label disagreements and raw probability differences. "
        "Some full-budget ReLU fits follow different numerical paths.")

nb.compare("MLPRegressor test predictions", regression_predictions,
           nb.results["mlp_regressor"]["predictions"], rtol=2e-12, atol=2e-12,
           reason="Float64 reduction and Adam update rounding")
nb.report()
nb.assert_parity()

| Dataset | α | sklearn accuracy | Moxie accuracy | Test label differences | Max test probability difference | Max mesh probability difference |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Moons | 0.1 | 0.875 | 0.875 | 0 | 0.0297 | 0.0881 |
| Moons | 0.316 | 0.950 | 0.950 | 0 | 0.0164 | 0.0885 |
| Moons | 1 | 0.950 | 0.950 | 0 | 0.00981 | 0.0522 |
| Moons | 3.16 | 0.875 | 0.875 | 0 | 1.48e-13 | 1.7e-13 |
| Moons | 10 | 0.475 | 0.475 | 0 | 2.22e-16 | 1.67e-16 |
| Circles | 0.1 | 0.800 | 0.775 | 1 | 0.198 | 0.465 |
| Circles | 0.316 | 0.875 | 0.875 | 0 | 0.148 | 0.279 |
| Circles | 1 | 0.900 | 0.900 | 0 | 0.00549 | 0.013 |
| Circles | 3.16 | 0.400 | 0.400 | 0 | 1.11e-16 | 1.11e-16 |
| Circles | 10 | 0.400 | 0.400 | 0 | 2.22e-16 | 1.67e-16 |
| Linear | 0.1 | 0.875 | 0.875 | 0 | 0.0668 | 0.614 |
| Linear | 0.316 | 0.950 | 0.950 | 0 | 0.0376 | 0.111 |
| Linear | 1 | 0.950 | 0.950 | 0 | 0.00185 | 0.00404 |
| Linear | 3.16 | 0.925 | 0.925 | 0 | 6.99e-10 | 1.49e-09 |
| Linear | 10 | 0.825 | 0.825 | 0 | 0.000369 | 0.00037 |

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
StandardScaler step (moons),PASS,"(40, 2)","(40, 2)",0,0,"rtol=1e-12, atol=1e-14",,Float64 mean and variance reductions
StandardScaler step (circles),PASS,"(40, 2)","(40, 2)",0,0,"rtol=1e-12, atol=1e-14",,Float64 mean and variance reductions
StandardScaler step (linearly separable),PASS,"(40, 2)","(40, 2)",0,0,"rtol=1e-12, atol=1e-14",,Float64 mean and variance reductions
Panel 1 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",5.55e-16,5.66e-14,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 2 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",4.44e-16,2.41e-15,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 3 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",3.33e-16,1.68e-15,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 4 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",9.99e-16,9.3e-15,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 5 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",2.22e-16,4.38e-16,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 6 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",1.22e-15,3.08e-15,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
Panel 7 six-iteration test probabilities,PASS,"(40, 2)","(40, 2)",2.22e-16,6.58e-16,"rtol=2e-10, atol=2e-11",,Float64 reductions and six L-BFGS iteration updates
